In [ ]:
# Manual Canny Edge Detection in Python
import cv2
import numpy as np
import matplotlib.pyplot as plt


img = cv2.imread('downlaods/img.jpg', cv2.IMREAD_GRAYSCALE)


def gaussian_blur(image, kernel_size=5, sigma=1):
    return cv2.GaussianBlur(image, (kernel_size, kernel_size), sigma)

blurred = gaussian_blur(img)

def compute_gradients(image):
    Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
    Ky = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]])

    Ix = cv2.filter2D(image, -1, Kx)
    Iy = cv2.filter2D(image, -1, Ky)

    G = np.hypot(Ix, Iy)       
    G = G / G.max() * 255      
    theta = np.arctan2(Iy, Ix)

    return (G, theta)

gradient_magnitude, gradient_direction = compute_gradients(blurred)


def non_max_suppression(G, theta):
    M, N = G.shape 
    output = np.zeros((M, N), dtype=np.uint8) 
    angle = theta * 180. / np.pi 
    angle[angle < 0] += 180 

    for i in range(1, M-1):
        for j in range(1, N-1):
            q = 255
            r = 255

            if (0 <= angle[i,j] < 22.5) or (157.5 <= angle[i,j] <= 180):
                q = G[i, j+1]
                r = G[i, j-1]

            elif (22.5 <= angle[i,j] < 67.5):
                q = G[i+1, j-1]
                r = G[i-1, j+1]

            elif (67.5 <= angle[i,j] < 112.5):
                q = G[i+1, j]
                r = G[i-1, j]

            elif (112.5 <= angle[i,j] < 157.5):
                q = G[i-1, j-1]
                r = G[i+1, j+1]

            if (G[i,j] >= q) and (G[i,j] >= r):
                output[i,j] = G[i,j]
            else:
                output[i,j] = 0
    return output

nms = non_max_suppression(gradient_magnitude, gradient_direction)


def hysteresis(img, low_thresh=20, high_thresh=40):
    M, N = img.shape
    result = np.zeros((M, N), dtype=np.uint8)

    strong = 255 
    weak = 75

    strong_i, strong_j = np.where(img >= high_thresh)
    weak_i, weak_j = np.where((img <= high_thresh) & (img >= low_thresh))

    result[strong_i, strong_j] = strong
    result[weak_i, weak_j] = weak


    for i in range(1, M-1):
        for j in range(1, N-1):
            if result[i,j] == weak:
                if ((result[i+1, j-1] == strong) or (result[i+1, j] == strong) or (result[i+1, j+1] == strong)
                    or (result[i, j-1] == strong) or (result[i, j+1] == strong)
                    or (result[i-1, j-1] == strong) or (result[i-1, j] == strong) or (result[i-1, j+1] == strong)):
                    result[i, j] = strong
                else:
                    result[i, j] = 0
    return result

canny_edges = hysteresis(nms)



plt.title("Original")
plt.imshow(img, cmap='gray')
plt.show()


plt.imshow(gradient_magnitude,'gray')
plt.title("Gradient Magnitude")
plt.show()

plt.title("Canny Edge Detection (Manual)")
plt.imshow(canny_edges, cmap='gray')
plt.show()
